In [0]:
CATALOG = "cricket"

spark.sql(f"""
WITH k AS (
  SELECT p.person_id
  FROM {CATALOG}.gold.dim_player p
  WHERE p.canonical_name = 'V Kohli'
)
SELECT
  m.match_format,
  count(DISTINCT f.match_id)                              AS matches,
  sum(f.runs_batter)                                     AS runs,
  sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END)        AS balls_faced,
  round(100.0*sum(f.runs_batter)/
        nullif(sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END),0),2) AS strike_rate,
  sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
  sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
FROM {CATALOG}.gold.fact_ball f
JOIN k ON f.batter_id = k.person_id
JOIN {CATALOG}.gold.dim_match m ON f.match_id = m.match_id
WHERE NOT f.is_super_over
GROUP BY m.match_format
ORDER BY runs DESC
""").show()

In [0]:
CATALOG = "cricket"

spark.sql(f"""
WITH k AS (
  SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name = 'AK Markram'
),
-- batting contributions per innings (absent if run out without facing)
bat AS (
  SELECT
    f.match_id, f.innings_number,
    sum(f.runs_batter)                                       AS runs,
    sum(CASE WHEN f.is_ball_faced THEN 1 ELSE 0 END)          AS balls,
    sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
  FROM {CATALOG}.gold.fact_ball f
  JOIN k ON f.batter_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON f.match_id = m.match_id
  WHERE m.event_name = 'Indian Premier League' AND NOT f.is_super_over
  GROUP BY f.match_id, f.innings_number
),
-- every innings he was dismissed in (catches run-out-without-facing)
dism AS (
  SELECT DISTINCT w.match_id, w.innings_number
  FROM {CATALOG}.silver.wicket w
  JOIN k ON w.player_out_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON w.match_id = m.match_id
  JOIN {CATALOG}.silver.dim_innings i
       ON w.match_id = i.match_id AND w.innings_number = i.innings_number
  WHERE m.event_name = 'Indian Premier League'
    AND NOT i.is_super_over            -- <-- match the bat side's super-over exclusion
    AND w.kind NOT IN ('retired hurt', 'retired not out')
),
-- an INNINGS exists if he batted a ball OR was dismissed
innings_keys AS (
  SELECT match_id, innings_number FROM bat
  UNION
  SELECT match_id, innings_number FROM dism
),
batting AS (
  SELECT
    ik.match_id, ik.innings_number,
    COALESCE(b.runs,0)  AS runs,
    COALESCE(b.balls,0) AS balls,
    COALESCE(b.fours,0) AS fours,
    COALESCE(b.sixes,0) AS sixes,
    (d.match_id IS NOT NULL) AS was_out
  FROM innings_keys ik
  LEFT JOIN bat  b ON ik.match_id=b.match_id AND ik.innings_number=b.innings_number
  LEFT JOIN dism d ON ik.match_id=d.match_id AND ik.innings_number=d.innings_number
),
played AS (
  SELECT count(DISTINCT mp.match_id) AS mat
  FROM {CATALOG}.silver.match_player mp
  JOIN k ON mp.person_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON mp.match_id = m.match_id
  WHERE m.event_name = 'Indian Premier League'
    AND mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
)
SELECT
  (SELECT mat FROM played)                                   AS matches,
  count(*)                                                   AS innings,
  sum(CASE WHEN NOT was_out THEN 1 ELSE 0 END)               AS not_outs,
  sum(runs)                                                  AS runs,
  max(runs)                                                  AS high_score,
  round(sum(runs) / nullif(sum(CASE WHEN was_out THEN 1 ELSE 0 END),0), 2) AS average,
  sum(balls)                                                 AS balls_faced,
  round(100.0 * sum(runs) / nullif(sum(balls),0), 2)          AS strike_rate,
  sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS hundreds,
  sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS fifties,
  sum(fours)                                                 AS fours,
  sum(sixes)                                                 AS sixes
FROM batting
""").show()

In [0]:
CATALOG = "cricket"

spark.sql(f"""
WITH k AS (
  SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name = 'RG Sharma'
),
bat AS (
  SELECT
    s.season, f.match_id, f.innings_number,
    sum(f.runs_batter)                                       AS runs,
    sum(CASE WHEN f.is_ball_faced THEN 1 ELSE 0 END)          AS balls,
    sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
  FROM {CATALOG}.gold.fact_ball f
  JOIN k ON f.batter_id = k.person_id
  JOIN {CATALOG}.gold.dim_match  m ON f.match_id = m.match_id
  JOIN {CATALOG}.gold.dim_series s ON f.series_key = s.series_key
  WHERE m.event_name = 'Indian Premier League' AND NOT f.is_super_over
  GROUP BY s.season, f.match_id, f.innings_number
),
dism AS (
  SELECT DISTINCT m.season, w.match_id, w.innings_number
  FROM {CATALOG}.silver.wicket w
  JOIN k ON w.player_out_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON w.match_id = m.match_id
  JOIN {CATALOG}.silver.dim_innings i
       ON w.match_id = i.match_id AND w.innings_number = i.innings_number
  WHERE m.event_name = 'Indian Premier League'
    AND NOT i.is_super_over
    AND w.kind NOT IN ('retired hurt', 'retired not out')
),
innings_keys AS (
  SELECT season, match_id, innings_number FROM bat
  UNION
  SELECT season, match_id, innings_number FROM dism
),
batting AS (
  SELECT
    ik.season, ik.match_id, ik.innings_number,
    COALESCE(b.runs,0)  AS runs,
    COALESCE(b.balls,0) AS balls,
    COALESCE(b.fours,0) AS fours,
    COALESCE(b.sixes,0) AS sixes,
    (d.match_id IS NOT NULL) AS was_out
  FROM innings_keys ik
  LEFT JOIN bat  b ON ik.match_id=b.match_id AND ik.innings_number=b.innings_number
  LEFT JOIN dism d ON ik.match_id=d.match_id AND ik.innings_number=d.innings_number
),
played AS (
  SELECT m.season, count(DISTINCT mp.match_id) AS mat
  FROM {CATALOG}.silver.match_player mp
  JOIN k ON mp.person_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON mp.match_id = m.match_id
  WHERE m.event_name = 'Indian Premier League'
    AND mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
  GROUP BY m.season
)
SELECT
  b.season,
  p.mat                                                     AS mat,
  count(*)                                                  AS inns,
  sum(CASE WHEN NOT b.was_out THEN 1 ELSE 0 END)            AS no,
  sum(b.runs)                                               AS runs,
  max(b.runs)                                               AS hs,
  round(sum(b.runs)/nullif(sum(CASE WHEN b.was_out THEN 1 ELSE 0 END),0),2) AS avg,
  sum(b.balls)                                              AS bf,
  round(100.0*sum(b.runs)/nullif(sum(b.balls),0),2)         AS sr,
  sum(CASE WHEN b.runs>=100 THEN 1 ELSE 0 END)             AS h100,
  sum(CASE WHEN b.runs>=50 AND b.runs<100 THEN 1 ELSE 0 END) AS f50,
  sum(b.fours)                                             AS fours,
  sum(b.sixes)                                             AS sixes
FROM batting b
LEFT JOIN played p ON b.season = p.season
GROUP BY b.season, p.mat
ORDER BY b.season DESC
""").show(30)

In [0]:
CATALOG = "cricket"

spark.sql(f"""
WITH k AS (
  SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name = 'AK Markram'
),
bat AS (
  SELECT
    m.event_name,
    f.match_id, f.innings_number,
    sum(f.runs_batter)                                       AS runs,
    sum(CASE WHEN f.is_ball_faced THEN 1 ELSE 0 END)          AS balls,
    sum(CASE WHEN f.runs_batter=4 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS fours,
    sum(CASE WHEN f.runs_batter=6 AND f.non_boundary IS NOT TRUE THEN 1 ELSE 0 END) AS sixes
  FROM {CATALOG}.gold.fact_ball f
  JOIN k ON f.batter_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON f.match_id = m.match_id
  WHERE m.match_format = 'T20' AND m.is_international = false
    AND NOT f.is_super_over
  GROUP BY m.event_name, f.match_id, f.innings_number
),
dism AS (
  SELECT DISTINCT m.event_name, w.match_id, w.innings_number
  FROM {CATALOG}.silver.wicket w
  JOIN k ON w.player_out_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON w.match_id = m.match_id
  JOIN {CATALOG}.silver.dim_innings i ON w.match_id=i.match_id AND w.innings_number=i.innings_number
  WHERE m.match_format = 'T20' AND m.is_international = false
    AND NOT i.is_super_over
    AND w.kind NOT IN ('retired hurt', 'retired not out')     -- not dismissals
),
innings_keys AS (
  SELECT event_name, match_id, innings_number FROM bat
  UNION
  SELECT event_name, match_id, innings_number FROM dism
),
batting AS (
  SELECT
    ik.event_name, ik.match_id, ik.innings_number,
    COALESCE(b.runs,0)  AS runs,  COALESCE(b.balls,0) AS balls,
    COALESCE(b.fours,0) AS fours, COALESCE(b.sixes,0) AS sixes,
    (d.match_id IS NOT NULL) AS was_out
  FROM innings_keys ik
  LEFT JOIN bat  b ON ik.match_id=b.match_id AND ik.innings_number=b.innings_number
  LEFT JOIN dism d ON ik.match_id=d.match_id AND ik.innings_number=d.innings_number
),
played AS (
  SELECT m.event_name, count(DISTINCT mp.match_id) AS mat
  FROM {CATALOG}.silver.match_player mp
  JOIN k ON mp.person_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON mp.match_id = m.match_id
  WHERE m.match_format = 'T20' AND m.is_international = false
    AND mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
  GROUP BY m.event_name
)
SELECT
  b.event_name                                              AS tournament,
  p.mat                                                     AS mat,
  count(*)                                                  AS inns,
  sum(CASE WHEN NOT b.was_out THEN 1 ELSE 0 END)            AS no,
  sum(b.runs)                                               AS runs,
  max(b.runs)                                               AS hs,
  round(sum(b.runs)/nullif(sum(CASE WHEN b.was_out THEN 1 ELSE 0 END),0),2) AS avg,
  sum(b.balls)                                              AS bf,
  round(100.0*sum(b.runs)/nullif(sum(b.balls),0),2)         AS sr,
  sum(CASE WHEN b.runs>=100 THEN 1 ELSE 0 END)             AS h100,
  sum(CASE WHEN b.runs>=50 AND b.runs<100 THEN 1 ELSE 0 END) AS f50,
  sum(b.fours)                                             AS fours,
  sum(b.sixes)                                             AS sixes
FROM batting b
LEFT JOIN played p ON b.event_name = p.event_name
GROUP BY b.event_name, p.mat
ORDER BY runs DESC
""").show(30, truncate=False)

In [0]:
CATALOG = "cricket"

spark.sql(f"""
WITH k AS (
  SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name = 'JR Hazlewood'
),
-- per bowling innings
bowl_inns AS (
  SELECT
    m.event_name, f.match_id, f.innings_number,
    sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END)                        AS legal_balls,
    sum(f.runs_batter + f.extra_wides + f.extra_noballs + f.extra_penalty)  AS runs_conceded,
    sum(CASE WHEN f.is_bowler_wicket THEN 1 ELSE 0 END)                      AS wickets
  FROM {CATALOG}.gold.fact_ball f
  JOIN k ON f.bowler_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON f.match_id = m.match_id
  WHERE m.match_format = 'T20' AND m.is_international = false
    AND NOT f.is_super_over
  GROUP BY m.event_name, f.match_id, f.innings_number
),
-- per match (for BBM / 10w): combine a bowler's innings within a match
bowl_match AS (
  SELECT event_name, match_id,
         sum(wickets) AS m_wkts, sum(runs_conceded) AS m_runs
  FROM bowl_inns GROUP BY event_name, match_id
),
-- BBI: rank innings by (wickets desc, runs asc), pick the best per tournament
bbi_rank AS (
  SELECT event_name, wickets, runs_conceded,
         row_number() OVER (PARTITION BY event_name
                            ORDER BY wickets DESC, runs_conceded ASC) AS rn
  FROM bowl_inns
),
-- BBM: same but on match totals
bbm_rank AS (
  SELECT event_name, m_wkts, m_runs,
         row_number() OVER (PARTITION BY event_name
                            ORDER BY m_wkts DESC, m_runs ASC) AS rn
  FROM bowl_match
),
played AS (
  SELECT m.event_name, count(DISTINCT mp.match_id) AS mat
  FROM {CATALOG}.silver.match_player mp
  JOIN k ON mp.person_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON mp.match_id = m.match_id
  WHERE m.match_format = 'T20' AND m.is_international = false
    AND mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
  GROUP BY m.event_name
),
agg AS (
  SELECT
    bi.event_name,
    count(*)                                            AS inns,
    sum(bi.legal_balls)                                 AS balls,
    sum(bi.runs_conceded)                               AS runs,
    sum(bi.wickets)                                     AS wkts,
    sum(CASE WHEN bi.wickets >= 4 THEN 1 ELSE 0 END)    AS four_w,
    sum(CASE WHEN bi.wickets >= 5 THEN 1 ELSE 0 END)    AS five_w
  FROM bowl_inns bi GROUP BY bi.event_name
),
tenw AS (
  SELECT event_name, sum(CASE WHEN m_wkts >= 10 THEN 1 ELSE 0 END) AS ten_w
  FROM bowl_match GROUP BY event_name
)
SELECT
  a.event_name                                          AS tournament,
  p.mat                                                 AS mat,
  a.inns                                                AS inns,
  a.balls                                               AS balls,
  a.runs                                                AS runs,
  a.wkts                                                AS wkts,
  concat(bbi.wickets, '/', bbi.runs_conceded)           AS bbi,
  concat(bbm.m_wkts,  '/', bbm.m_runs)                  AS bbm,
  round(a.runs*1.0/nullif(a.wkts,0), 2)                 AS ave,
  round(6.0*a.runs/nullif(a.balls,0), 2)                AS econ,
  round(a.balls*1.0/nullif(a.wkts,0), 1)                AS sr,
  a.four_w                                              AS `4w`,
  a.five_w                                              AS `5w`,
  t.ten_w                                               AS `10w`
FROM agg a
LEFT JOIN played p ON a.event_name = p.event_name
LEFT JOIN bbi_rank bbi ON a.event_name = bbi.event_name AND bbi.rn = 1
LEFT JOIN bbm_rank bbm ON a.event_name = bbm.event_name AND bbm.rn = 1
LEFT JOIN tenw t      ON a.event_name = t.event_name
ORDER BY wkts DESC
""").show(30, truncate=False)

In [0]:
CATALOG = "cricket"

spark.sql(f"""
SELECT person_id, canonical_name FROM {CATALOG}.gold.dim_player WHERE canonical_name LIKE '%Hazlewood%'
""").show(30, truncate=False)

In [0]:
CATALOG = "cricket"

spark.sql(f"""
WITH k AS (
  SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name = 'JR Hazlewood'
),
bowl_inns AS (
  SELECT
    s.season, f.match_id, f.innings_number,
    sum(CASE WHEN f.is_legal_ball THEN 1 ELSE 0 END)                        AS legal_balls,
    sum(f.runs_batter + f.extra_wides + f.extra_noballs + f.extra_penalty)  AS runs_conceded,
    sum(CASE WHEN f.is_bowler_wicket THEN 1 ELSE 0 END)                      AS wickets
  FROM {CATALOG}.gold.fact_ball f
  JOIN k ON f.bowler_id = k.person_id
  JOIN {CATALOG}.gold.dim_match  m ON f.match_id = m.match_id
  JOIN {CATALOG}.gold.dim_series s ON f.series_key = s.series_key
  WHERE m.event_name = 'Indian Premier League' AND NOT f.is_super_over
  GROUP BY s.season, f.match_id, f.innings_number
),
bowl_match AS (
  SELECT season, match_id, sum(wickets) AS m_wkts, sum(runs_conceded) AS m_runs
  FROM bowl_inns GROUP BY season, match_id
),
bbi_rank AS (
  SELECT season, wickets, runs_conceded,
         row_number() OVER (PARTITION BY season
                            ORDER BY wickets DESC, runs_conceded ASC) AS rn
  FROM bowl_inns
),
bbm_rank AS (
  SELECT season, m_wkts, m_runs,
         row_number() OVER (PARTITION BY season
                            ORDER BY m_wkts DESC, m_runs ASC) AS rn
  FROM bowl_match
),
played AS (
  SELECT m.season, count(DISTINCT mp.match_id) AS mat
  FROM {CATALOG}.silver.match_player mp
  JOIN k ON mp.person_id = k.person_id
  JOIN {CATALOG}.gold.dim_match m ON mp.match_id = m.match_id
  WHERE m.event_name = 'Indian Premier League'
    AND mp.match_id NOT IN (SELECT match_id FROM {CATALOG}.silver.excluded_match)
  GROUP BY m.season
),
agg AS (
  SELECT
    season,
    count(*)                                            AS inns,
    sum(legal_balls)                                    AS balls,
    sum(runs_conceded)                                  AS runs,
    sum(wickets)                                        AS wkts,
    sum(CASE WHEN wickets >= 4 THEN 1 ELSE 0 END)       AS fourw,
    sum(CASE WHEN wickets >= 5 THEN 1 ELSE 0 END)       AS fivew
  FROM bowl_inns GROUP BY season
),
tenw AS (
  SELECT season, sum(CASE WHEN m_wkts >= 10 THEN 1 ELSE 0 END) AS tenw
  FROM bowl_match GROUP BY season
)
SELECT
  a.season                                              AS season,
  p.mat                                                 AS mat,
  a.inns                                                AS inns,
  a.balls                                               AS balls,
  a.runs                                                AS runs,
  a.wkts                                                AS wkts,
  concat(bbi.wickets, '/', bbi.runs_conceded)           AS bbi,
  concat(bbm.m_wkts,  '/', bbm.m_runs)                  AS bbm,
  round(a.runs*1.0/nullif(a.wkts,0), 2)                 AS ave,
  round(6.0*a.runs/nullif(a.balls,0), 2)                AS econ,
  round(a.balls*1.0/nullif(a.wkts,0), 1)                AS sr,
  a.fourw                                               AS fourw,
  a.fivew                                               AS fivew,
  t.tenw                                                AS tenw
FROM agg a
LEFT JOIN played p ON a.season = p.season
LEFT JOIN bbi_rank bbi ON a.season = bbi.season AND bbi.rn = 1
LEFT JOIN bbm_rank bbm ON a.season = bbm.season AND bbm.rn = 1
LEFT JOIN tenw t       ON a.season = t.season
ORDER BY a.season DESC
""").show(30, truncate=False)

In [0]:
CATALOG = "cricket"
spark.sql(f"""
WITH k AS (SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name='RG Sharma')
SELECT w.kind, count(*) AS cnt
FROM {CATALOG}.silver.wicket w
JOIN k ON w.player_out_id = k.person_id
JOIN {CATALOG}.gold.dim_match m ON w.match_id = m.match_id
WHERE m.event_name='Indian Premier League' AND m.season='2026'
GROUP BY w.kind
""").show()

In [0]:
CATALOG = "cricket"
spark.sql(f"""
WITH k AS (SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name='KL Rahul'),
squad AS (   -- matches he was in the XI (2022 IPL)
  SELECT DISTINCT mp.match_id
  FROM {CATALOG}.silver.match_player mp JOIN k ON mp.person_id=k.person_id
  JOIN {CATALOG}.gold.dim_match m ON mp.match_id=m.match_id
  WHERE m.event_name='Indian Premier League' AND m.season='2022'
),
batted AS (  -- matches he actually faced a ball in
  SELECT DISTINCT f.match_id
  FROM {CATALOG}.gold.fact_ball f JOIN k ON f.batter_id=k.person_id
  WHERE NOT f.is_super_over
)
SELECT s.match_id, m.team_a, m.team_b, m.start_date, m.outcome_result, m.won_by_team
FROM squad s
JOIN {CATALOG}.gold.dim_match m ON s.match_id=m.match_id
WHERE s.match_id NOT IN (SELECT match_id FROM batted)
""").show(truncate=False)

In [0]:
CATALOG = "cricket"
spark.sql(f"""
WITH k AS (SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name='KL Rahul')
SELECT
  mp.match_id,
  m.start_date,
  m.team_a, m.team_b,
  m.outcome_result,
  m.won_by_team,
  m.scheduled_overs,
  -- how many balls actually bowled in this match (abandoned games are short)
  (SELECT count(*) FROM {CATALOG}.gold.fact_ball f WHERE f.match_id = mp.match_id) AS balls
FROM {CATALOG}.silver.match_player mp
JOIN k ON mp.person_id = k.person_id
JOIN {CATALOG}.gold.dim_match m ON mp.match_id = m.match_id
WHERE m.event_name='Indian Premier League' AND m.season='2025'
ORDER BY m.start_date
""").show(30, truncate=False)

In [0]:
CATALOG = "cricket"
spark.sql(f"""
WITH k AS (SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name='KL Rahul')
SELECT 'bat' AS src, f.match_id, f.innings_number, f.is_super_over
FROM {CATALOG}.gold.fact_ball f JOIN k ON f.batter_id=k.person_id
JOIN {CATALOG}.gold.dim_match m ON f.match_id=m.match_id
WHERE m.event_name='Indian Premier League' AND m.season='2020/21'
GROUP BY f.match_id, f.innings_number, f.is_super_over

UNION ALL

SELECT 'dism', w.match_id, w.innings_number, i.is_super_over
FROM {CATALOG}.silver.wicket w JOIN k ON w.player_out_id=k.person_id
JOIN {CATALOG}.gold.dim_match m ON w.match_id=m.match_id
LEFT JOIN {CATALOG}.silver.dim_innings i
  ON w.match_id=i.match_id AND w.innings_number=i.innings_number
WHERE m.event_name='Indian Premier League' AND m.season='2020/21'
ORDER BY match_id, innings_number, src
""").show(50, truncate=False)

In [0]:
spark.sql(f"""
WITH k AS (SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name='KL Rahul')
SELECT w.match_id, w.innings_number, w.kind, w.player_out
FROM {CATALOG}.silver.wicket w JOIN k ON w.player_out_id=k.person_id
JOIN {CATALOG}.gold.dim_match m ON w.match_id=m.match_id
WHERE m.event_name='Indian Premier League' AND m.season='2022'
  AND w.match_id NOT IN (
    SELECT DISTINCT f.match_id FROM {CATALOG}.gold.fact_ball f
    JOIN k ON f.batter_id=k.person_id
  )
""").show(truncate=False)

In [0]:
CATALOG = "cricket"
spark.sql(f"""
WITH k AS (SELECT person_id FROM {CATALOG}.gold.dim_player WHERE canonical_name='V Kohli')
SELECT
  s.season,
  sum(f.runs_batter) AS runs,
  -- balls faced = everything except wides (no-balls ARE faced)
  sum(CASE WHEN f.extra_wides = 0 THEN 1 ELSE 0 END)                       AS balls_faced,
  round(100.0*sum(f.runs_batter)/
        nullif(sum(CASE WHEN f.extra_wides = 0 THEN 1 ELSE 0 END),0),2)     AS strike_rate
FROM {CATALOG}.gold.fact_ball f
JOIN k ON f.batter_id = k.person_id
JOIN {CATALOG}.gold.dim_match  m ON f.match_id = m.match_id
JOIN {CATALOG}.gold.dim_series s ON f.series_key = s.series_key
WHERE m.event_name='Indian Premier League' AND NOT f.is_super_over
GROUP BY s.season ORDER BY s.season DESC
""").show(30)

In [0]:
spark.sql(f"""
SELECT
  t.team_name,
  count(*)                                                        AS matches,
  sum(CASE WHEN m.won_by_team = t.team_name THEN 1 ELSE 0 END)     AS won,
  sum(CASE WHEN m.outcome_result = 'tie' THEN 1 ELSE 0 END)        AS tied,
  sum(CASE WHEN m.outcome_result = 'no result' THEN 1 ELSE 0 END)  AS no_result
FROM {CATALOG}.gold.dim_match m
JOIN {CATALOG}.silver.match_player mp ON m.match_id = mp.match_id
JOIN {CATALOG}.gold.dim_team t ON mp.team = t.team_name
WHERE m.match_type = 'IT20'          -- T20Is only (not league T20)
GROUP BY t.team_name
HAVING count(*) > 20
ORDER BY matches DESC
""").show(30)

In [0]:
CATALOG = "cricket"

# which silver tables exist yet, and how big — shows how far the run has progressed
expected = ["dim_match","match_registry","deliveries","player_name","dim_player",
            "dim_team","dim_series","dim_calendar","dim_venue",
            "dim_phase","ball_phase","wicket","wicket_fielder",
            "match_player","dim_innings"]

existing = {t.name for t in spark.catalog.listTables(f"{CATALOG}.silver")}

for t in expected:
    if t in existing:
        try:
            n = spark.table(f"{CATALOG}.silver.{t}").count()
            print(f"✓ {t:16} {n:>12,}")
        except Exception as e:
            print(f"~ {t:16} (writing…)")
    else:
        print(f"·  {t:16} (not yet)")

In [0]:
CATALOG = "cricket"
for t in ["wicket","match_player","dim_innings","deliveries"]:
    d = spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.{t} LIMIT 1").select("timestamp","operation").collect()[0]
    n = spark.table(f"{CATALOG}.silver.{t}").select("match_id").distinct().count()
    print(f"{t:16} | matches={n:>6} | last_written={d['timestamp']}")

In [0]:
CATALOG = "cricket"
print("fact_ball:", spark.table(f"{CATALOG}.gold.fact_ball").count())
print("deliveries:", spark.table(f"{CATALOG}.silver.deliveries").count())

In [0]:
import os
JSON_DIR = "/Volumes/cricket/bronze/landing/json"
files = [f for f in os.listdir(JSON_DIR) if f.endswith(".json")]
print("JSON files in landing:", len(files))

In [0]:
# Kohli IPL by season — now a simple GROUP BY, no CTEs, no conventions to remember
spark.sql(f"""
SELECT
  season,
  count(*)                                          AS inns,
  sum(CASE WHEN not_out THEN 1 ELSE 0 END)          AS no,
  sum(runs)                                         AS runs,
  max(runs)                                         AS hs,
  round(sum(runs)/nullif(sum(CASE WHEN was_out THEN 1 ELSE 0 END),0),2) AS avg,
  sum(balls_faced)                                  AS bf,
  round(100.0*sum(runs)/nullif(sum(balls_faced),0),2) AS sr,
  sum(CASE WHEN runs>=100 THEN 1 ELSE 0 END)        AS h100,
  sum(CASE WHEN runs>=50 AND runs<100 THEN 1 ELSE 0 END) AS f50,
  sum(fours)                                        AS fours,
  sum(sixes)                                        AS sixes
FROM {CATALOG}.gold.batting_innings bi
JOIN {CATALOG}.gold.dim_player p ON bi.person_id = p.person_id
WHERE p.canonical_name = 'V Kohli' AND bi.event_name = 'Indian Premier League'
GROUP BY season ORDER BY season DESC
""").show(30)